In [3]:
from pathlib import Path
import math
import pyarrow as pa
import pyarrow.parquet as pq

inp = Path("C:/Users/insoo/Documents/Western_WAI/explainable-misinfo-ai/data/processed/fakenewsnet/fakenewsnet_true.parquet")
out_dir = inp.parent
prefix = "fakenewsnet_true_part"
target_mb = 45  # aim under 50MB

# cleanup any old parts
for p in out_dir.glob(prefix + "*.parquet"):
    p.unlink()

pf = pq.ParquetFile(inp)
total_rows = pf.metadata.num_rows
total_bytes = inp.stat().st_size
rows_per_part = max(1, int(total_rows * (target_mb * 1024 * 1024) / total_bytes))
schema = pf.schema_arrow

print(f"Input: {inp}  size={total_bytes/1_048_576:.2f}MB  rows={total_rows}")
print(f"Target ~{target_mb}MB => rows_per_part≈{rows_per_part}")

part_idx = 1
rows_in_part = 0
writer = None

def open_writer(i: int):
    out = out_dir / f"{prefix}{i}.parquet"
    # compression helps keep parts smaller
    return out, pq.ParquetWriter(out, schema, compression="snappy")

out_path, writer = open_writer(part_idx)

def write_table(tbl: pa.Table):
    global writer
    writer.write_table(tbl)

for batch in pf.iter_batches(batch_size=65536):
    rb = batch
    start = 0
    while start < rb.num_rows:
        remaining = rows_per_part - rows_in_part
        take = min(remaining, rb.num_rows - start)
        piece = rb.slice(start, take)
        write_table(pa.Table.from_batches([piece]))
        rows_in_part += take
        start += take

        if rows_in_part >= rows_per_part:
            writer.close()
            part_size = out_path.stat().st_size / 1_048_576
            print(f"Wrote {out_path.name}: {part_size:.2f} MB")
            part_idx += 1
            rows_in_part = 0
            out_path, writer = open_writer(part_idx)

# close last writer if it has data
writer.close()
if out_path.exists():
    part_size = out_path.stat().st_size / 1_048_576
    print(f"Wrote {out_path.name}: {part_size:.2f} MB")

# list all parts
parts = sorted(out_dir.glob(prefix + "*.parquet"))
print("\nParts:")
for p in parts:
    print(f"  {p.name}: {p.stat().st_size/1_048_576:.2f} MB")


Input: C:\Users\insoo\Documents\Western_WAI\explainable-misinfo-ai\data\processed\fakenewsnet\fakenewsnet_true.parquet  size=71.23MB  rows=12537
Target ~45MB => rows_per_part≈7920
Wrote fakenewsnet_true_part1.parquet: 45.99 MB
Wrote fakenewsnet_true_part2.parquet: 25.17 MB

Parts:
  fakenewsnet_true_part1.parquet: 45.99 MB
  fakenewsnet_true_part2.parquet: 25.17 MB
